[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/10_Image_Embeddings.ipynb)

# Embeddings de imágenes — CLIP

En **07** y **08** vimos embeddings de palabras (Word2Vec), y en **09** embeddings de texto con un modelo preentrenado (Gemini). Aquí damos un paso más: **una imagen también puede volverse un vector**.

Usamos **CLIP** (OpenAI, 2021): un modelo entrenado con 400 millones de pares imagen-texto de internet. Lo interesante no es solo que embebe imágenes — es que **mete texto e imágenes en el mismo espacio vectorial**. La foto de un gato y la frase "a photo of a cat" terminan con vectores parecidos.

Vamos en 5 pasos:

1. Cargar CLIP desde Hugging Face.
2. Embedding de una imagen.
3. Embedding de texto — mismo espacio, mismas dimensiones.
4. Comparar imagen y texto con similitud coseno.
5. Clasificación **zero-shot**: la imagen elige, entre varias frases, cuál la describe mejor — sin haber sido entrenada para esas categorías.

**Nota:** esta es una primera versión del notebook; la iremos afinando.

## Archivos que necesitas

Ninguno. Las imágenes de ejemplo se descargan solas desde Wikimedia Commons.

## Setup — instalar librerías

`transformers` trae CLIP listo para usar; no hace falta entrenar nada.

In [ ]:
%pip install -q transformers torch pillow numpy matplotlib requests

**GPU:** no es necesaria. El modelo (~150 MB) corre rápido en CPU. Si tienes GPU activada en Colab, se usa sola.

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

plt.rcParams.update({"figure.dpi": 100, "axes.grid": False})

## 0 — Imágenes de ejemplo

Cuatro fotos de Wikimedia Commons, una por categoría: gato, perro, bicicleta y pizza.

In [ ]:
IMAGENES = {
    "gato": "https://commons.wikimedia.org/wiki/Special:FilePath/Cat03.jpg?width=400",
    "perro": "https://commons.wikimedia.org/wiki/Special:FilePath/Golde33443.jpg?width=400",
    "bicicleta": "https://commons.wikimedia.org/wiki/Special:FilePath/Bicycle.jpg?width=400",
    "pizza": "https://commons.wikimedia.org/wiki/Special:FilePath/Eq_it-na_pizza-margherita_sep2005_sml.jpg?width=400",
}

def descargar_imagen(url):
    """Baja una imagen y la abre con PIL, sin guardarla a disco."""
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    return Image.open(io.BytesIO(resp.content)).convert("RGB")

fotos = {nombre: descargar_imagen(url) for nombre, url in IMAGENES.items()}

fig, axes = plt.subplots(1, len(fotos), figsize=(11, 3))
for ax, (nombre, img) in zip(axes, fotos.items()):
    ax.imshow(img)
    ax.set_title(nombre)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 1 — Cargar CLIP

`openai/clip-vit-base-patch32`: un encoder de imágenes (Vision Transformer) y un encoder de texto, entrenados juntos para que la imagen y su descripción caigan cerca en el mismo espacio de 512 dimensiones.

In [ ]:
MODELO_CLIP = "openai/clip-vit-base-patch32"

modelo = CLIPModel.from_pretrained(MODELO_CLIP)
procesador = CLIPProcessor.from_pretrained(MODELO_CLIP)

print(f"Modelo listo: {MODELO_CLIP}")

## 2 — Embedding de una imagen

Le pasamos la foto del gato y obtenemos su vector de 512 números.

In [ ]:
entradas = procesador(images=fotos["gato"], return_tensors="pt")

with torch.no_grad():
    vector_gato = modelo.get_image_features(**entradas)[0]

print(f"Dimensiones del vector: {vector_gato.shape[0]}")
print(f"Primeros 8 valores: {vector_gato[:8].numpy().round(3)}")

## 3 — Embedding de texto (mismo espacio)

Ahora le pasamos frases en inglés — CLIP se entrenó sobre todo con captions en ese idioma, así que funciona mejor ahí. Fíjate que el vector de texto tiene **las mismas 512 dimensiones** que el de la imagen: viven en el mismo espacio.

In [ ]:
textos = ["a photo of a cat", "a photo of a dog", "a photo of a bicycle", "a photo of a pizza"]

entradas_texto = procesador(text=textos, return_tensors="pt", padding=True)

with torch.no_grad():
    vectores_texto = modelo.get_text_features(**entradas_texto)

print(f"Shape: {vectores_texto.shape}  (4 frases, 512 dimensiones cada una)")

## 4 — ¿Qué tan parecido es el texto a la imagen?

Misma similitud coseno de siempre. Esperamos que `"a photo of a cat"` sea la frase más parecida al vector de la foto del gato.

In [ ]:
def similitud_coseno(u, v):
    u, v = u.detach().numpy(), v.detach().numpy()
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

print(f"{'frase':>28s}    sim_coseno")
for texto, vec_texto in zip(textos, vectores_texto):
    sim = similitud_coseno(vector_gato, vec_texto)
    print(f"  {texto!r:>26s}:  {sim:+.3f}")

## 5 — Clasificación zero-shot

"Zero-shot" = el modelo nunca fue entrenado para distinguir estas 4 categorías puntuales, pero puede **comparar** la imagen contra cualquier frase que se nos ocurra. Embebemos las 4 fotos y las 4 frases, armamos la matriz de similitud, y por cada imagen gana la frase más parecida.

In [ ]:
entradas_imgs = procesador(images=list(fotos.values()), return_tensors="pt")
with torch.no_grad():
    vectores_imgs = modelo.get_image_features(**entradas_imgs)

# Matriz de similitud: filas = imágenes, columnas = frases
matriz = np.zeros((len(fotos), len(textos)))
for i, vi in enumerate(vectores_imgs):
    for j, vt in enumerate(vectores_texto):
        matriz[i, j] = similitud_coseno(vi, vt)

nombres_fotos = list(fotos.keys())
print(f"{'imagen':>12s}   {'predicción':>24s}   sim_coseno")
for i, nombre in enumerate(nombres_fotos):
    j = int(np.argmax(matriz[i]))
    marca = "✓" if nombre in textos[j] else "✗"
    print(f"{nombre:>12s}   {textos[j]!r:>24s}  {marca}  {matriz[i, j]:+.3f}")

In [ ]:
azul = plt.cm.colors.LinearSegmentedColormap.from_list("azul", ["#ffffff", "#2a78d6"])

fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(matriz, cmap=azul, vmin=0, vmax=matriz.max())

ax.set_xticks(range(len(textos)))
ax.set_xticklabels(textos, rotation=30, ha="right")
ax.set_yticks(range(len(nombres_fotos)))
ax.set_yticklabels(nombres_fotos)

for i in range(len(nombres_fotos)):
    for j in range(len(textos)):
        color = "white" if matriz[i, j] > matriz.max() * 0.6 else "#333333"
        ax.text(j, i, f"{matriz[i, j]:.2f}", ha="center", va="center", color=color, fontsize=9)

ax.set_title("Similitud coseno: imagen × frase")
fig.colorbar(im, ax=ax, label="similitud coseno")
plt.tight_layout()
plt.show()

## Cierre

Con el mismo truco de siempre (vectores + similitud coseno) comparamos ahora **modalidades distintas**: una foto contra una frase. Esa es la idea detrás de la búsqueda "texto → imagen" (como en Google Fotos o Pinterest) y de los modelos que generan imágenes a partir de texto.

**Límites de esta primera versión:**

- CLIP funciona mejor con texto en **inglés** — en español la similitud baja un poco.
- Con solo 4 categorías, el zero-shot es casi trivial. Se pone interesante con docenas de candidatos o categorías ambiguas.
- No hay negative sampling ni entrenamiento aquí: CLIP ya viene entrenado, solo lo usamos.